# Task A — BAZA comparison (Wave 1)

**Uruchom: Kernel → Restart & Run All** (kolejność komórek ma znaczenie).

Notebook:
1. pobiera 48 runów z W&B (`group=TaskA_BAZA`);
2. sprawdza komplet 6 × 2 × 4 i zamrożony `candidate_fingerprint`;
3. wybiera głębokość **tylko na `clean` / validation**;
4. porównuje scenariusze przy zamrożonej głębokości;
5. zapisuje CSV do `outputs/taskA_BAZA_analysis/`.

Kolejność wyboru L\*: max valid AUPRC → niższy Brier → niższy cosine oversmoothing → mniej warstw.

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from IPython.display import display

warnings.filterwarnings("ignore", category=Warning)

# Resolve paths relative to this notebook / repo root (not the process cwd).
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "TaskA_BAZA_comparison.ipynb").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "notebooks" / "TaskA_BAZA_comparison.ipynb").exists():
    REPO_ROOT = NOTEBOOK_DIR
else:
    # Fallback: walk up looking for configs/ + src/
    REPO_ROOT = NOTEBOOK_DIR
    for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (parent / "configs").is_dir() and (parent / "src").is_dir():
            REPO_ROOT = parent
            break

OUT_DIR = REPO_ROOT / "outputs" / "taskA_BAZA_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ENTITY = "politechnika-gnn-thesis"
PROJECT = "politechnika-gnn-thesis"
GROUP = "TaskA_BAZA"

SCENARIOS = [
    "clean",
    "hidden_confounder",
    "selection_bias",
    "no_overlap",
    "noisy_documentation",
    "multihospital",
]
MODELS = ["hetero_sage", "rgcn"]
LAYERS = [1, 2, 3, 4]
EXPECTED = len(SCENARIOS) * len(MODELS) * len(LAYERS)

print("REPO_ROOT:", REPO_ROOT)
print("OUT_DIR:  ", OUT_DIR)

api = wandb.Api()
runs = list(api.runs(f"{ENTITY}/{PROJECT}", filters={"group": GROUP}))
print(f"Fetched runs in group={GROUP}: {len(runs)}")

In [ ]:
def cfg_get(cfg: dict, *keys, default=None):
    """Read nested Hydra/W&B config; also accepts flat dotted keys."""
    cur = cfg
    ok = True
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            ok = False
            break
        cur = cur[key]
    if ok:
        return cur

    dotted = ".".join(keys)
    if dotted in cfg:
        return cfg[dotted]
    return default


def summary_get(summary: dict, *names, default=np.nan):
    for name in names:
        if name in summary and summary[name] is not None:
            return summary[name]
    return default


def find_oversmoothing_col(columns, *preferred_suffixes):
    cols = [str(c) for c in columns]
    for suffix in preferred_suffixes:
        for col in cols:
            if col == suffix or col.endswith(suffix) or suffix in col:
                return col
    return None


rows = []
for run in runs:
    cfg = dict(run.config or {})
    summary = {str(k): v for k, v in dict(run.summary or {}).items()}

    scenario = cfg_get(cfg, "data", "dataset", "scenario")
    model = cfg_get(cfg, "model", "conv_type")
    if model is None:
        model = cfg_get(cfg, "model", "name")
    layers = cfg_get(cfg, "model", "num_layers")

    row = {
        "run_id": run.id,
        "run_name": run.name,
        "state": run.state,
        "scenario": scenario,
        "model": model,
        "num_layers": int(layers) if layers is not None else None,
        "training_seed": cfg_get(cfg, "training", "seed"),
        "candidate_seed": cfg_get(cfg, "data", "candidate_seed"),
        "candidate_fingerprint": summary_get(
            summary, "candidate_fingerprint", default=None
        ),
        "best_epoch": summary_get(summary, "best_epoch"),
        "classification_threshold": summary_get(
            summary, "classification_threshold"
        ),
        "valid_auprc": summary_get(
            summary, "valid_auprc", "best_valid_AUPRC", "best_valid_metric"
        ),
        "valid_auc": summary_get(summary, "valid_auc"),
        "valid_brier": summary_get(summary, "valid_brier"),
        "valid_loss": summary_get(summary, "valid_loss"),
        "valid_f1": summary_get(summary, "valid_f1"),
        "valid_precision": summary_get(summary, "valid_precision"),
        "valid_recall": summary_get(summary, "valid_recall"),
        "valid_auprc_baseline": summary_get(summary, "valid_auprc_baseline"),
        "valid_auprc_lift": summary_get(summary, "valid_auprc_lift"),
        "test_auprc": summary_get(summary, "test_auprc"),
        "test_auc": summary_get(summary, "test_auc"),
        "test_brier": summary_get(summary, "test_brier"),
        "test_loss": summary_get(summary, "test_loss"),
        "test_f1": summary_get(summary, "test_f1"),
        "test_precision": summary_get(summary, "test_precision"),
        "test_recall": summary_get(summary, "test_recall"),
        "test_auprc_baseline": summary_get(summary, "test_auprc_baseline"),
        "test_auprc_lift": summary_get(summary, "test_auprc_lift"),
    }

    for key, value in summary.items():
        if "oversmoothing" in key and isinstance(value, (int, float, np.floating)):
            row[key] = float(value)

    rows.append(row)

df = pd.DataFrame(rows)
print("df shape:", df.shape)
display(df.head())

In [ ]:
finished = df[df["state"] == "finished"].copy()
print(f"finished: {len(finished)} / expected: {EXPECTED}")

missing = []
for scenario in SCENARIOS:
    for model in MODELS:
        for layers in LAYERS:
            hit = finished[
                (finished["scenario"] == scenario)
                & (finished["model"] == model)
                & (finished["num_layers"] == layers)
            ]
            if hit.empty:
                missing.append((scenario, model, layers))

print("missing combos:", len(missing))
if missing:
    display(pd.DataFrame(missing, columns=["scenario", "model", "num_layers"]))
else:
    print("Complete matrix 6 × 2 × 4.")

fps = finished["candidate_fingerprint"].dropna().astype(str).unique()
print("unique candidate fingerprints:", list(fps))
if len(fps) == 0:
    print("WARNING: no candidate_fingerprint in summaries.")
elif len(fps) > 1:
    print("WARNING: candidate sets differ — scenario comparison is compromised.")
else:
    print("OK: all finished runs share one candidate fingerprint.")

In [ ]:
def pick_depth_on_clean(model_name: str) -> int:
    """Select num_layers on clean using validation only."""
    sub = finished[
        (finished["scenario"] == "clean") & (finished["model"] == model_name)
    ].copy()
    if sub.empty:
        raise ValueError(f"No finished clean runs for model={model_name}")

    cosine_col = find_oversmoothing_col(
        sub.columns,
        "oversmoothing/cosine_sim_mean",
        "cosine_sim_mean",
    )

    sub["_auprc"] = pd.to_numeric(sub["valid_auprc"], errors="coerce")
    sub["_brier"] = pd.to_numeric(sub["valid_brier"], errors="coerce")
    sub["_cos"] = (
        pd.to_numeric(sub[cosine_col], errors="coerce") if cosine_col else np.nan
    )
    sub["_layers"] = pd.to_numeric(sub["num_layers"], errors="coerce")

    sub = sub.sort_values(
        by=["_auprc", "_brier", "_cos", "_layers"],
        ascending=[False, True, True, True],
        na_position="last",
    )
    chosen = int(sub.iloc[0]["num_layers"])
    print(f"{model_name} → L{chosen}" + (f" (cosine col: {cosine_col})" if cosine_col else ""))
    display(
        sub[["num_layers", "valid_auprc", "valid_brier", "test_auprc"]]
        .reset_index(drop=True)
    )
    return chosen


chosen_depth = {model: pick_depth_on_clean(model) for model in MODELS}
print("Frozen depths:", chosen_depth)

In [ ]:
# Analysis A — validation curves vs depth
cosine_col = find_oversmoothing_col(
    finished.columns,
    "oversmoothing/cosine_sim_mean",
    "cosine_sim_mean",
)

for model in MODELS:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True)
    for scenario in SCENARIOS:
        sub = finished[
            (finished["model"] == model) & (finished["scenario"] == scenario)
        ].sort_values("num_layers")
        axes[0].plot(sub["num_layers"], sub["valid_auprc"], marker="o", label=scenario)
        axes[1].plot(sub["num_layers"], sub["valid_brier"], marker="o", label=scenario)
        if cosine_col is not None:
            axes[2].plot(sub["num_layers"], sub[cosine_col], marker="o", label=scenario)
    axes[0].set_title(f"{model}: valid AUPRC vs layers")
    axes[1].set_title(f"{model}: valid Brier vs layers")
    axes[2].set_title(f"{model}: oversmoothing cosine vs layers")
    for ax in axes:
        ax.set_xlabel("num_layers")
        ax.set_xticks(LAYERS)
        ax.grid(True, alpha=0.3)
    axes[0].legend(fontsize=7, loc="best")
    if cosine_col is None:
        axes[2].text(0.5, 0.5, "no oversmoothing column", ha="center", va="center")
    plt.tight_layout()
    plt.show()

In [ ]:
# Analysis B — frozen architecture chosen on clean
frozen = pd.concat(
    [
        finished[
            (finished["model"] == model) & (finished["num_layers"] == depth)
        ].copy()
        for model, depth in chosen_depth.items()
    ],
    ignore_index=True,
)

pivot_valid = frozen.pivot_table(
    index="scenario", columns="model", values="valid_auprc"
).reindex(SCENARIOS)
pivot_test = frozen.pivot_table(
    index="scenario", columns="model", values="test_auprc"
).reindex(SCENARIOS)

print("Frozen depths:", chosen_depth)
print("\nValid AUPRC (frozen depth)")
display(pivot_valid)
print("Test AUPRC (frozen depth) — report only after selection on valid")
display(pivot_test)

drops = []
for model in MODELS:
    clean_val = float(pivot_valid.loc["clean", model])
    for scenario in SCENARIOS:
        valid_auprc = float(pivot_valid.loc[scenario, model])
        drops.append(
            {
                "model": model,
                "scenario": scenario,
                "num_layers_frozen": chosen_depth[model],
                "valid_auprc": valid_auprc,
                "delta_vs_clean_valid": valid_auprc - clean_val,
                "test_auprc": float(pivot_test.loc[scenario, model]),
            }
        )

drop_df = pd.DataFrame(drops)
display(drop_df)

In [ ]:
# Heatmaps: scenario × layers (valid AUPRC)
for model in MODELS:
    mat = (
        finished[finished["model"] == model]
        .pivot_table(index="scenario", columns="num_layers", values="valid_auprc")
        .reindex(index=SCENARIOS, columns=LAYERS)
    )
    print(model)
    display(mat.style.background_gradient(axis=None).format("{:.3f}"))

In [ ]:
# Export CSV — requires previous cells (use Restart & Run All)
required = {
    "finished": "finished",
    "frozen": "frozen",
    "drop_df": "drop_df",
    "OUT_DIR": "OUT_DIR",
}
missing_names = [name for name, var in required.items() if var not in globals()]
if missing_names:
    raise RuntimeError(
        "Brakuje zmiennych: "
        + ", ".join(missing_names)
        + ". Uruchom Kernel → Restart & Run All od pierwszej komórki."
    )

results_path = OUT_DIR / "taskA_BAZA_results.csv"
frozen_path = OUT_DIR / "taskA_BAZA_frozen_depth.csv"
drop_path = OUT_DIR / "taskA_BAZA_scenario_drop.csv"

finished.to_csv(results_path, index=False)
frozen.to_csv(frozen_path, index=False)
drop_df.to_csv(drop_path, index=False)

print("Wrote:")
print(" ", results_path.resolve())
print(" ", frozen_path.resolve())
print(" ", drop_path.resolve())
print("\nChosen depths:", chosen_depth)